# 02 — Visualization & Trend Analysis

**Dataset:** Pengangguran Menurut Golongan Umur (2021–2025)  
**Source:** BPS — Badan Pusat Statistik (Statistics Indonesia)  
**Source URL:** https://www.bps.go.id/  
**Data date:** 2021 Februari – 2025 Agustus  
**Purpose:** Produce final interactive charts and static figures for reporting and Kaggle submission.

## Imports

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
print('Libraries loaded.')

## Configuration / Constants

In [ ]:
DATA_DIR = (
    Path('/kaggle/input/unemployment-indonesia')
    if Path('/kaggle').exists()
    else Path('../data/processed')
)
OUTPUTS_DIR = Path('../outputs') if not Path('/kaggle').exists() else Path('/kaggle/working')
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

AGE_GROUP_ORDER = ['15-19', '20-24', '25-29', '30-34', '35-39', '40-44', '45-49', '50-54', '55-59', '60+']
PERIOD_ORDER = ['Februari', 'Agustus']
SOURCE_NOTE = 'Source: BPS — Badan Pusat Statistik (Statistics Indonesia)'

# Plotly discrete sequence — colorblind-friendly
PLOTLY_COLORS = px.colors.qualitative.Safe

print(f'DATA_DIR: {DATA_DIR.resolve()}')
print(f'OUTPUTS_DIR: {OUTPUTS_DIR.resolve()}')

## Data Loading

Load the consolidated long-format CSV produced by notebook 01_eda.ipynb.

In [ ]:
df = pd.read_csv(DATA_DIR / 'unemployment_combined.csv')

age_cat = pd.CategoricalDtype(categories=AGE_GROUP_ORDER, ordered=True)
period_cat = pd.CategoricalDtype(categories=PERIOD_ORDER, ordered=True)
df['age_group'] = df['age_group'].astype(str).astype(age_cat)
df['period'] = df['period'].astype(str).astype(period_cat)
df.sort_values(['year', 'period', 'age_group'], inplace=True)
df.reset_index(drop=True, inplace=True)

print(f'Loaded: {df.shape}')
df.head()

In [ ]:
# Build stable time labels for x-axis ordering
period_abbr = {'Februari': 'Feb', 'Agustus': 'Aug'}
period_sort = {'Februari': 0, 'Agustus': 1}

df['time_label'] = df['year'].astype(str) + '-' + df['period'].astype(str).map(period_abbr)
df['_sort_key'] = df['year'].astype(int) * 10 + df['period'].astype(str).map(period_sort)

time_order = (
    df[['time_label', '_sort_key']].drop_duplicates()
    .sort_values('_sort_key')['time_label']
    .tolist()
)
print('Time axis order:', time_order)

## Visualization

All charts use **Plotly** for interactive display in Kaggle / Jupyter.
Each figure is also exported to `outputs/` as a standalone HTML file.

### Chart 1: Unemployment Trend by Age Group

In [ ]:
fig1 = px.line(
    df.sort_values('_sort_key'),
    x='time_label',
    y='jumlah',
    color='age_group',
    category_orders={'time_label': time_order, 'age_group': AGE_GROUP_ORDER},
    markers=True,
    color_discrete_sequence=PLOTLY_COLORS,
)
fig1.update_layout(
    title='Unemployment by Age Group Over Time (2021–2025)',
    xaxis_title='Survey Period',
    yaxis_title='Jumlah Pengangguran (Orang)',
    legend_title='Age Group',
    yaxis_tickformat=',',
    annotations=[{'text': SOURCE_NOTE, 'xref': 'paper', 'yref': 'paper',
                  'x': 0.5, 'y': -0.15, 'showarrow': False, 'font': {'size': 10, 'color': 'gray'}}],
)
fig1.write_html(str(OUTPUTS_DIR / 'interactive_trend.html'))
fig1.show()

### Chart 2: February vs. August Total Unemployment per Year

In [ ]:
agg = df.groupby(['year', 'period'], observed=True)['jumlah'].sum().reset_index()
agg['period'] = agg['period'].astype(str)

fig2 = px.bar(
    agg,
    x='year',
    y='jumlah',
    color='period',
    barmode='group',
    category_orders={'period': PERIOD_ORDER},
    color_discrete_sequence=PLOTLY_COLORS[:2],
)
fig2.update_layout(
    title='Total Unemployment: February vs. August per Year (2021–2025)',
    xaxis_title='Year',
    yaxis_title='Jumlah Pengangguran (Orang)',
    legend_title='Survey Period',
    yaxis_tickformat=',',
    annotations=[{'text': SOURCE_NOTE, 'xref': 'paper', 'yref': 'paper',
                  'x': 0.5, 'y': -0.15, 'showarrow': False, 'font': {'size': 10, 'color': 'gray'}}],
)
fig2.write_html(str(OUTPUTS_DIR / 'feb_vs_aug.html'))
fig2.show()

### Chart 3: Unemployment Heatmap (Age Group × Survey Period)

In [ ]:
pivot = df.pivot_table(index='age_group', columns='time_label', values='jumlah', aggfunc='sum')
pivot = pivot.loc[AGE_GROUP_ORDER, time_order]

fig3 = px.imshow(
    pivot,
    color_continuous_scale='YlOrRd',
    text_auto=True,
    aspect='auto',
)
fig3.update_layout(
    title='Unemployment Heatmap: Age Group × Survey Period (2021–2025)',
    xaxis_title='Survey Period',
    yaxis_title='Age Group',
    coloraxis_colorbar_title='Orang',
    annotations=[{'text': SOURCE_NOTE, 'xref': 'paper', 'yref': 'paper',
                  'x': 0.5, 'y': -0.12, 'showarrow': False, 'font': {'size': 10, 'color': 'gray'}}],
)
fig3.write_html(str(OUTPUTS_DIR / 'heatmap.html'))
fig3.show()

### Chart 4: Pernah vs. Tidak Pernah Bekerja Composition by Age Group

In [ ]:
comp = df.groupby('age_group', observed=True)[['pernah_bekerja', 'tidak_pernah_bekerja']].mean().reset_index()
comp['age_group'] = comp['age_group'].astype(str)
comp_melt = comp.melt(id_vars='age_group', var_name='category', value_name='mean_count')
label_map = {'pernah_bekerja': 'Pernah Bekerja', 'tidak_pernah_bekerja': 'Tidak Pernah Bekerja'}
comp_melt['category'] = comp_melt['category'].map(label_map)

fig4 = px.bar(
    comp_melt,
    x='age_group',
    y='mean_count',
    color='category',
    barmode='stack',
    category_orders={'age_group': AGE_GROUP_ORDER},
    color_discrete_sequence=PLOTLY_COLORS[:2],
)
fig4.update_layout(
    title='Composition: Previously Employed vs. Never Employed by Age Group (Average)',
    xaxis_title='Age Group',
    yaxis_title='Mean Jumlah Pengangguran (Orang)',
    legend_title='Employment History',
    yaxis_tickformat=',',
    annotations=[{'text': SOURCE_NOTE, 'xref': 'paper', 'yref': 'paper',
                  'x': 0.5, 'y': -0.15, 'showarrow': False, 'font': {'size': 10, 'color': 'gray'}}],
)
fig4.write_html(str(OUTPUTS_DIR / 'composition_by_age.html'))
fig4.show()

### Chart 5: Year-over-Year Total Unemployment Change

In [ ]:
annual = df.groupby(['year', 'period'], observed=True)['jumlah'].sum().reset_index()
annual['period_str'] = annual['period'].astype(str)
annual.sort_values(['period_str', 'year'], inplace=True)
annual['yoy_pct'] = annual.groupby('period_str')['jumlah'].pct_change() * 100

fig5 = px.bar(
    annual.dropna(subset=['yoy_pct']),
    x='year',
    y='yoy_pct',
    color='period_str',
    barmode='group',
    category_orders={'period_str': PERIOD_ORDER},
    color_discrete_sequence=PLOTLY_COLORS[:2],
)
fig5.add_hline(y=0, line_dash='dash', line_color='black', line_width=1)
fig5.update_layout(
    title='Year-over-Year Change in Total Unemployment (%)',
    xaxis_title='Year',
    yaxis_title='YoY Change (%)',
    legend_title='Survey Period',
    annotations=[{'text': SOURCE_NOTE, 'xref': 'paper', 'yref': 'paper',
                  'x': 0.5, 'y': -0.15, 'showarrow': False, 'font': {'size': 10, 'color': 'gray'}}],
)
fig5.write_html(str(OUTPUTS_DIR / 'yoy_change.html'))
fig5.show()

## Summary / Conclusions

**Key visual insights:**

1. **20–24 dominance**: The 20–24 age group is the single largest contributor to unemployment across all years and periods, consistently 2–3× larger than adjacent cohorts.
2. **Post-2021 recovery**: Total unemployment declined sharply from the August 2021 peak (~9.1 M) through 2023, then stabilised around 7.3–7.5 M in 2024–2025.
3. **February vs. August asymmetry**: August figures are generally higher than February, likely driven by mid-year graduation waves inflating the 20–24 and 25–29 cohorts.
4. **Never-employed concentration**: Stacked bars confirm that younger cohorts (15–19, 20–24) skew heavily toward first-time job seekers; older cohorts skew toward previously employed.
5. **60+ volatility**: The heatmap highlights the 60+ group’s large swings between surveys, signalling sensitivity to how informal and agricultural work is captured each wave.
6. **Structural floor**: YoY change lines flatten near 0% in 2024–2025, consistent with the economy approaching a structural unemployment floor.

> **Note:** All figures are national-level aggregates. Regional or gender-based conclusions cannot be drawn from this dataset.